Denoising Diffusion Implicit Model(DDIM, 去噪扩散隐式模型)

- 基于DDPM，改进：
    1. 加速采样：修改采样公式，去噪时'跳步骤'，在一次去噪迭代中直接预测若干次去噪后的结果；
    2. 变更采样方差：推广了前向加噪过程和反向去噪过程，可以自定义模型的噪声强度，让同一个训练好的 DDPM 有不同的采样效果
- 在DDPM中，贝叶斯公式求解：
$$q(x_{t-1} | x_t,x_0) = q(x_t | x_{t-1},x_0)\frac{q(x_{t-1}|x_0)}{q(x_t|x_0)}$$
其中，右式三项全部都已知的，故可以求出左式<u>*【在'01_diffusion_model.ipynb'中已推导】*</u>。另外，还知道$q(x_t | x_{t-1},x_0)$在前向过程中可以推出$x_t = \sqrt{\bar\alpha_t}x_0 + \sqrt{1 - \bar\alpha_t}\epsilon$。因此，发现 DDPM 的训练目标实际上只约束了 $q(x_t|x_0)$ ，而并没有强制要求 $x_t \leftrightarrow x_{t-1}$ 必须是某个特定的马尔可夫链。所以，只要满足 $q(x_t|x_0)$ 不变，中间路径其实可以改。

  \**DDPM 对应的是一种扩散路径，而不是唯一的扩散路径。*
- 构造：$$x_{t-1} = \sqrt{\bar\alpha_{t-1}}\hat x_0 + \sqrt{1 - \bar\alpha_{t-1}}\epsilon_{\theta} \tag{*}$$
其中无随机项，所以 $x_t \to x_{t-1}$ 的去噪过程，变成了 $x_{t-1} = f(x_t)$ 的一个确定性函数表达（*即采样轨迹从DDPM的随机轨迹变成了确定性轨迹*）。

    代入：将 $\hat x_0 = \frac{x_t - \sqrt{1 - \bar \alpha_t}\epsilon_{\theta}}{\sqrt{\bar \alpha_t}}$ 代入 $(\ast)$式 中，得到完整采样公式：
$$x_{t-1} = \sqrt{\bar\alpha_{t-1}} (\frac{x_t - \sqrt{1 - \bar \alpha_t}\epsilon_{\theta}}{\sqrt{\bar \alpha_t}}) + \sqrt{1 - \bar\alpha_{t-1} - \sigma^2}\epsilon_{\theta} + \sigma \epsilon$$

---

**\[注1\]** 为了方差匹配，所以进行构造:
> 原方差为：$Var(x_{t-1}|x_0)=1-\bar\alpha_{t-1}$；\
> DDIM将噪声拆分为两部分：$A\epsilon_{\theta}+\sigma\epsilon$，其中 $\epsilon$是新设的一个标准正态分布，相当于重新随机采样的噪声，而$\epsilon_{\theta}$是网络预测的噪声；\
> 于是，现方差为：$Var(x_{t-1}|x_0)=A^2 + \sigma^2$；\
> 令两式相等，解得：$A = \sqrt{1 - \bar\alpha_{t-1} - \sigma^2}$

---

- 可用任意的 $x_{prev}$ 代替 $x_{t-1}$，变形为：
$$x_{prev} = \sqrt{\bar\alpha_{prev}} (\frac{x_t - \sqrt{1 - \bar \alpha_t}\epsilon_{\theta}}{\sqrt{\bar \alpha_t}}) + \sqrt{1 - \bar\alpha_{prev} - \sigma^2}\epsilon_{\theta} + \sigma \epsilon$$
其中 $x_{prev}$ 和 $x_{t-1}$ 可以相隔多个迭代步数。

---

**\[注2\]** 引入自由参数 $\eta$ 控制随机性：
> 在上式中，$\sigma$ 并没有被确定下来，只需要满足 $0 \leq \sigma^2 \leq 1-\bar\alpha_{t-1}$ 即可成；\
> 故引入：$\sigma = \eta \sqrt{\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}} \sqrt{1- \frac{\bar\alpha_t}{\bar\alpha_{t-1}}} = \eta \sqrt{\tilde\beta_t}$\
> （$\sqrt{\tilde\beta_t}$ 是DDPM后验方差的标准差）\
> 当 $\eta=1$ 时，$\sigma=\sqrt{\tilde\beta_t}$，此时为DDPM；\
> 当 $\eta=0$ 时，$\sigma=0$，随机项消失，此时为DDIM。\
> 实际上，选择不同的 $\eta$ 是在DDPM和DDIM之间插值，$\eta$控制了插值的比例。

---

**总结**：

DDPM学习“如何去噪”，DDIM发现“去噪路径不唯一”，于是将随机马尔科夫链改成了可跳步骤的确定性轨迹，从而大幅提升反向过程中的采样速度，且并不修改前向训练过程。